In [ ]:
"""
Hotel Booking Cancellation Predictor — Deep Learning Version
DL Lab Final Project

Concepts applied:
  - Multi-layer feedforward neural network (MLP)
  - Batch Normalization
  - Dropout regularization
  - Early Stopping & Learning Rate Scheduling
  - Binary Cross-Entropy loss + Adam optimizer
  - Feature Engineering
  - Gradio UI with model accuracy display
"""

import pandas as pd
import numpy as np
import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# ─────────────────────────────────────────────
# 1. LOAD & CLEAN DATA
# ─────────────────────────────────────────────
print("📂 Loading dataset …")
df = pd.read_csv("C:\\Users\\Asher\\Desktop\\hotel_booking.csv")

# Handle missing values
df['children']    = df['children'].fillna(0)
df['agent']       = df['agent'].fillna(0)
df['country']     = df['country'].fillna('Unknown')

# ─────────────────────────────────────────────
# 2. FEATURE ENGINEERING  (richer than ML version)
# ─────────────────────────────────────────────
df['total_guests']       = df['adults'] + df['children'] + df['babies']
df['total_nights']       = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['revenue_estimate']   = df['adr'] * df['total_nights'].replace(0, 1)
df['guest_per_night']    = df['total_guests'] / df['total_nights'].replace(0, 1)
df['has_previous_cancel'] = (df['previous_cancellations'] > 0).astype(int)
df['is_repeated_guest']  = df['is_repeated_guest'].fillna(0)

# Filter invalid rows
df = df[(df['total_guests'] > 0) & (df['total_nights'] > 0)]

features = [
    'lead_time',
    'total_guests',
    'total_nights',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'booking_changes',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'adr',
    'revenue_estimate',
    'guest_per_night',
    'has_previous_cancel',
    'is_repeated_guest',
    'days_in_waiting_list',
]

X = df[features].fillna(0).values
y = df['is_canceled'].values

print(f"✅ Dataset: {X.shape[0]:,} rows | {X.shape[1]} features")
print(f"   Cancellation rate: {y.mean()*100:.1f}%")

# ─────────────────────────────────────────────
# 3. TRAIN / VAL / TEST SPLIT
# ─────────────────────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f"\n📊 Split → Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# ─────────────────────────────────────────────
# 4. BUILD DEEP LEARNING MODEL
# ─────────────────────────────────────────────
def build_model(input_dim: int) -> keras.Model:
    """
    Deep MLP with:
      • 4 hidden layers (256 → 128 → 64 → 32)
      • Batch Normalization after each layer  ← DL concept
      • Dropout for regularization            ← DL concept
      • ReLU activations
      • Sigmoid output for binary classification
    """
    inp = keras.Input(shape=(input_dim,), name="input")

    x = layers.Dense(256, name="dense_1")(inp)
    x = layers.BatchNormalization(name="bn_1")(x)         # Batch Norm
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.3, name="drop_1")(x)             # Dropout

    x = layers.Dense(128, name="dense_2")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.3, name="drop_2")(x)

    x = layers.Dense(64, name="dense_3")(x)
    x = layers.BatchNormalization(name="bn_3")(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.2, name="drop_3")(x)

    x = layers.Dense(32, name="dense_4")(x)
    x = layers.BatchNormalization(name="bn_4")(x)
    x = layers.Activation("relu")(x)

    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = keras.Model(inputs=inp, outputs=out, name="HotelCancellationDNN")
    return model

model = build_model(X_train_s.shape[1])
model.summary()

# ─────────────────────────────────────────────
# 5. COMPILE  (Adam optimizer + BCE loss)
# ─────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),   # Adam      ← DL concept
    loss="binary_crossentropy",                             # BCE loss  ← DL concept
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

# ─────────────────────────────────────────────
# 6. CALLBACKS
# ─────────────────────────────────────────────
cb_early = callbacks.EarlyStopping(                        # Early Stopping ← DL concept
    monitor="val_loss", patience=8,
    restore_best_weights=True, verbose=1
)
cb_lr = callbacks.ReduceLROnPlateau(                       # LR Scheduling  ← DL concept
    monitor="val_loss", factor=0.5, patience=4,
    min_lr=1e-6, verbose=1
)
cb_ckpt = callbacks.ModelCheckpoint(
    "best_hotel_dl_model.keras",
    monitor="val_auc", mode="max",
    save_best_only=True, verbose=0
)

# ─────────────────────────────────────────────
# 7. TRAIN
# ─────────────────────────────────────────────
print("\n🚀 Training neural network …")
history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=60,
    batch_size=256,
    callbacks=[cb_early, cb_lr, cb_ckpt],
    verbose=1
)

# ─────────────────────────────────────────────
# 8. EVALUATE
# ─────────────────────────────────────────────
y_prob = model.predict(X_test_s, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
auc      = roc_auc_score(y_test, y_prob)

print(f"\n{'='*50}")
print(f"  Test Accuracy : {accuracy*100:.2f}%")
print(f"  Test ROC-AUC  : {auc:.4f}")
print(f"{'='*50}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=["Not Cancelled", "Cancelled"]))

# ─────────────────────────────────────────────
# 9. SAVE TRAINING CURVE PLOT
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Training History", fontsize=14, fontweight="bold")

axes[0].plot(history.history['loss'],     label='Train Loss', color='#E05C5C')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='#5CA8E0')
axes[0].set_title("Binary Cross-Entropy Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train Acc', color='#E05C5C')
axes[1].plot(history.history['val_accuracy'], label='Val Acc',   color='#5CA8E0')
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("📈 Training curves saved → training_curves.png")

# ─────────────────────────────────────────────
# 10. PREDICTION FUNCTION
# ─────────────────────────────────────────────
def predict_risk(lead_time, total_guests, total_nights,
                 previous_cancellations, previous_not_canceled,
                 booking_changes, car_parking, special_requests,
                 adr, is_repeated_guest):

    revenue_est  = adr * max(total_nights, 1)
    guest_p_night = total_guests / max(total_nights, 1)
    has_prev_cancel = int(previous_cancellations > 0)

    row = np.array([[
        lead_time, total_guests, total_nights,
        previous_cancellations, previous_not_canceled,
        booking_changes, car_parking, special_requests,
        adr, revenue_est, guest_p_night,
        has_prev_cancel, is_repeated_guest,
        0   # days_in_waiting_list (not in UI for simplicity)
    ]])

    row_scaled = scaler.transform(row)
    prob = float(model.predict(row_scaled, verbose=0)[0][0])

    risk_pct = prob * 100
    if risk_pct >= 65:
        emoji = "🔴"
        label = "HIGH CANCELLATION RISK"
    elif risk_pct >= 40:
        emoji = "🟡"
        label = "MEDIUM CANCELLATION RISK"
    else:
        emoji = "🟢"
        label = "LOW CANCELLATION RISK"

    return (
        f"{emoji} {label}\n\n"
        f"Cancellation probability : {risk_pct:.1f}%\n"
        f"Confidence (stay)         : {(1-prob)*100:.1f}%\n\n"
        f"Model: Deep Neural Network  |  "
        f"Test Accuracy: {accuracy*100:.2f}%  |  AUC: {auc:.3f}"
    )

# ─────────────────────────────────────────────
# 11. GRADIO UI
# ─────────────────────────────────────────────
with gr.Blocks(title="🏨 Hotel Cancellation DL Predictor") as app:

    gr.Markdown("""
    # 🏨 Hotel Booking Cancellation Predictor
    ### Deep Learning Version — DL Lab Final Project
    **Architecture:** 4-layer MLP · Batch Normalization · Dropout · Adam · Early Stopping
    """)

    with gr.Row():
        with gr.Column():
            gr.Markdown("#### 📅 Booking Details")
            lead_time          = gr.Number(label="Lead Time (days)", value=50)
            total_guests       = gr.Number(label="Total Guests", value=2)
            total_nights       = gr.Number(label="Total Nights", value=3)
            adr                = gr.Number(label="Average Daily Rate (ADR $)", value=100)

        with gr.Column():
            gr.Markdown("#### 📋 Guest History")
            prev_cancel        = gr.Number(label="Previous Cancellations", value=0)
            prev_not_canceled  = gr.Number(label="Previous Bookings (Not Cancelled)", value=0)
            is_repeated        = gr.Number(label="Repeated Guest (0/1)", value=0)

        with gr.Column():
            gr.Markdown("#### 🔧 Booking Extras")
            booking_changes    = gr.Number(label="Booking Changes", value=0)
            car_parking        = gr.Number(label="Car Parking Spaces", value=0)
            special_requests   = gr.Number(label="Special Requests", value=0)

    predict_btn = gr.Button("🔮 Predict Cancellation Risk", variant="primary", size="lg")
    output_box  = gr.Textbox(label="Prediction", lines=5, interactive=False)

    predict_btn.click(
        fn=predict_risk,
        inputs=[lead_time, total_guests, total_nights,
                prev_cancel, prev_not_canceled,
                booking_changes, car_parking, special_requests,
                adr, is_repeated],
        outputs=output_box
    )

    gr.Examples(
        examples=[
            [150, 2, 3, 0, 0, 0, 0, 0, 100, 0],
            [10,  4, 7, 2, 3, 1, 2, 2, 200, 0],
            [30,  2, 2, 0, 1, 1, 0, 1, 150, 1],
            [200, 1, 1, 3, 0, 0, 0, 0, 80,  0],
        ],
        inputs=[lead_time, total_guests, total_nights,
                prev_cancel, prev_not_canceled,
                booking_changes, car_parking, special_requests,
                adr, is_repeated],
        outputs=output_box,
        fn=predict_risk,
        cache_examples=False
    )

    gr.Markdown(f"""
    ---
    **DL Concepts Applied:**
    `Multi-layer Perceptron` · `Batch Normalization` · `Dropout` · 
    `Binary Cross-Entropy Loss` · `Adam Optimizer` · `Early Stopping` · 
    `ReduceLROnPlateau` · `Feature Engineering` · `Train/Val/Test Split`

    **Results →** Test Accuracy: **{accuracy*100:.2f}%** | ROC-AUC: **{auc:.4f}**
    """)

# ─────────────────────────────────────────────
# 12. LAUNCH
# ─────────────────────────────────────────────
if __name__ == "__main__":
    app.launch()
    if __name__ == "__main__":
        app.launch(share=True)


c:\Users\Asher\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📂 Loading dataset …
✅ Dataset: 118,565 rows | 14 features
   Cancellation rate: 37.3%

📊 Split → Train: 85,663 | Val: 15,117 | Test: 17,785


Model: "HotelCancellationDNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 14)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │         3,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_1 (Dropout)                │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_2 (Dropout)                │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_3 (BatchNormalization)       │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_3 (Dropout)                │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_4 (BatchNormalization)       │ (None, 32)             │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,025 (191.50 KB)

 Trainable params: 48,065 (187.75 KB)

 Non-trainable params: 960 (3.75 KB)


🚀 Training neural network …
Epoch 1/60
335/335 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.7409 - auc: 0.7885 - loss: 0.5172 - val_accuracy: 0.7585 - val_auc: 0.8127 - val_loss: 0.4920 - learning_rate: 0.0010
Epoch 2/60
335/335 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.7567 - auc: 0.8074 - loss: 0.4953 - val_accuracy: 0.7595 - val_auc: 0.8191 - val_loss: 0.4833 - learning_rate: 0.0010
Epoch 3/60
335/335 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.7596 - auc: 0.8133 - loss: 0.4889 - val_accuracy: 0.7582 - val_auc: 0.8207 - val_loss: 0.4811 - learning_rate: 0.0010
Epoch 4/60
335/335 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7623 - auc: 0.8165 - loss: 0.4857 - val_accuracy: 0.7619 - val_auc: 0.8230 - val_loss: 0.4784 - learning_rate: 0.0010
Epoch 5/60
335/335 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7640 - auc: 0.8176 - loss: 0.4841 - val_accuracy: 0.7615 - val_auc: 0.8244 - val_loss: 0.4773 - learning_rate: 0.0010
Epoch 6/60
335/335 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/ste